# Evaluating LM Outputs using Rubric + LM Judges

Instead of using BLADE's `EntireAnalysisProcessed` code, we will try an evaluation implementation that reads straight from `multirun_analyses.json` and gives the results to an LLM judge.

In [1]:
# imports
import yaml
import json
from stat_genie.blade_pipeline.llms.config import llm

In [2]:
# define file paths
multirun_load_path = "analysis_output/multirun_analyses.json"
llm_eval_config_path = "../../config/llm_eval_config.yml"

In [3]:
# load config and json
llm_eval_config = yaml.safe_load(open(llm_eval_config_path))
with open(multirun_load_path, "r") as file:
    analyses = json.load(file)

In [4]:
analyses['analyses']['0']['cvars'].keys()

dict_keys(['ivs', 'dv', 'controls'])

In [5]:
analyses['analyses']['0']['cvars']['controls']

[{'description': 'Maximum wind speed at landfall (measures storm strength/severity).',
  'is_moderator': False,
  'moderator_on': None,
  'columns': ['wind']},
 {'description': 'Saffir–Simpson category at landfall (ordinal measure of storm severity). Treated as a potential moderator: name femininity effects on deaths may be weaker/absent for very high-category storms because severity is salient regardless of name.',
  'is_moderator': True,
  'moderator_on': 'masfem_std',
  'columns': ['category']},
 {'description': 'Minimum central pressure at landfall (another objective measure of storm strength; lower = stronger).',
  'is_moderator': False,
  'moderator_on': None,
  'columns': ['min']},
 {'description': 'Calendar year (controls for secular trends in reporting, preparedness and infrastructure).',
  'is_moderator': False,
  'moderator_on': None,
  'columns': ['year']},
 {'description': 'Elapsed years since the hurricane (alternative control for reporting changes and retrospective adjus

In [6]:
# use llm interface to compare features
### BASIC EXAMPLE, PROMPT NEEDS LOTS OF WORK
prompt = f"""Given the following analysis results: \
    {analyses['analyses']['0']} and {analyses['analyses']['1']}, rate their \
        consistency on a scale of 1 to 5 for each of the following categories:
        1) Features X used in the model
        2) Output y in the model
        3) Model class/type
        4) Outcome result."""
gen = llm(provider="openai", model="gpt-5-mini")
response = gen.generate([{"role": "user", "content": prompt}])
response

[2025-11-10 08:29:37.02][config_load.py:34 - blade_bench.llms.config_load:load_config][INFO] Loaded config from '/accounts/grad/zachrewolinski/research/stat-genie/config/llm_config.yml'.
[2025-11-10 08:29:37.32][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-11-10 08:29:56.58][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  19.25 seconds
[2025-11-10 08:29:56.58][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)


TextGenResponse(text=[Message(role='assistant', content='Summary ratings (1 = very inconsistent, 5 = highly consistent)\n\n1) Features X used in the model: 4/5\n- Reason: Both analyses use the same core predictors (name femininity, a binary gender indicator, wind, category, min, year, elapsedyrs) and also include damage as a robustness outcome. Differences are mostly naming/standardization (masfem_std vs masfem_z), an extra MTurk femininity measure in the second, and the second includes a categorical source control while the first does not. The first also explicitly constructs an interaction (masfem × category) that the second does not. These are relatively minor but meaningful differences.\n\n2) Output y in the model: 2/5\n- Reason: The first analysis models raw alldeaths as a count outcome (Negative Binomial GLM) while the second models log_alldeaths (log(1+deaths)) via OLS. These are different dependent-variable formulations with different distributional assumptions and interpretati